In [1]:
%pip install -U huggingface_hub pandas pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:

from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
import shutil
import time
import os

DATASET = "IINOVAII/Insurance_Dataset"

OUTPUT_DIR = Path(
    r"D:\MASTERS\FOML\assignment\Project\Project@"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_FILE = OUTPUT_DIR / "insurance.csv"

api = HfApi()

# Find original CSV files in the dataset repository.
files = api.list_repo_files(
    repo_id=DATASET,
    repo_type="dataset"
)

csv_files = [
    name for name in files
    if name.lower().endswith(".csv")
]

if len(csv_files) != 1:
    raise RuntimeError(
        f"Expected exactly one original CSV, "
        f"but found {len(csv_files)}: {csv_files}. "
        "Cannot create one unmodified original CSV "
        "without changing the source files."
    )

original_name = csv_files[0]

if FINAL_FILE.exists():
    raise FileExistsError(
        f"File already exists: {FINAL_FILE}"
    )

# Download to the Hugging Face cache.
# Interrupted downloads can reuse cached data.
for attempt in range(1, 11):
    try:
        source = Path(
            hf_hub_download(
                repo_id=DATASET,
                filename=original_name,
                repo_type="dataset"
            )
        )
        break

    except Exception as error:
        if attempt == 10:
            raise

        delay = min(2 ** attempt, 60)
        print(
            f"Download attempt {attempt} failed: {error}"
        )
        print(f"Retrying in {delay} seconds...")
        time.sleep(delay)

# Copy original bytes without parsing or modifying data.
temp_file = OUTPUT_DIR / "insurance.csv.part"

with source.open("rb") as src, temp_file.open("wb") as dst:
    shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)
    dst.flush()
    os.fsync(dst.fileno())

# Verify the copied file is byte-for-byte identical.
import hashlib

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for block in iter(
            lambda: file.read(8 * 1024 * 1024), b""
        ):
            digest.update(block)
    return digest.hexdigest()

if sha256(source) != sha256(temp_file):
    raise RuntimeError(
        "File integrity check failed."
    )

temp_file.rename(FINAL_FILE)

print("Download completed successfully.")
print("File:", FINAL_FILE)
print("Size:", FINAL_FILE.stat().st_size, "bytes")
print("SHA256:", sha256(FINAL_FILE))


Download completed successfully.
File: D:\MASTERS\FOML\assignment\Project\Project@\insurance.csv
Size: 194100597 bytes
SHA256: 7bdc738808f8c2d8dd7476af25f330ec997b5d96d6e9c5a6dda340e19a058222
